# Narrativa de la comparación de jugadores con LLM

`NarradorComparacion.redactar()` es el único componente de LLM que llegó a producción -- redacta en prosa la comparación entre dos jugadores que ya calculó la app.

Regla del spec: el LLM nunca calcula un número, solo verbaliza lo que ya resolvieron los percentiles, el modelo de valor y el clustering. A diferencia de los otros notebooks, este no genera ningún artefacto nuevo -- el código real ya vive en `app/`, esto documenta su diseño y sirve para seguir iterando el prompt desde un solo lugar.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# app/ es un proyecto uv independiente (su propio .venv, ver AGENTS.md "Dos mundos de
# Python") -- no se instala en .venv312. Pero narrativa/, valuation/ y schemas/ no dependen
# de FastAPI (solo de pydantic, joblib, pandas, openai, todos ya en .venv312), así que se
# pueden importar directo agregando app/src al sys.path, sin arrastrar el servidor.
sys.path.insert(0, str(Path("..") / "app" / "src"))

load_dotenv("../.env")  # mismo .env de la raíz que usa Settings en app/src/app/core/config.py
client_api_key = os.getenv("OPENAI_API_KEY")
assert client_api_key, "Falta OPENAI_API_KEY en el .env de la raíz"


## 1. Dos jugadores reales

Usamos `dataset.parquet`, el mismo que sirve el backend -- ya trae percentiles y arquetipo precalculados, no hay que recalcular nada acá.

Elegimos dos delanteros centro de la temporada más reciente con valores de mercado bien distintos, para ver si el modelo + el LLM explican bien esa brecha.

In [2]:
df = pd.read_parquet("../data/processed/dataset.parquet")

recientes = df[df["Season"].astype(str) == "2025-2026"]
fila_a = recientes[recientes["Player"] == "Erling Haaland"].iloc[0]
fila_b = recientes[recientes["Player"] == "Kylian Mbappé"].iloc[0]

fila_a[["Player", "Squad", "posicion_tm", "Age", "valor_eur", "cluster_label"]]


Player                                Erling Haaland
Squad                                Manchester City
posicion_tm                           Centre-Forward
Age                                             25.0
valor_eur                                200000000.0
cluster_label    Estrella ofensiva de máximo volumen
Name: 11479, dtype: object

## 2. Armar la comparación con las clases reales

Nada se reimplementa a mano: son las mismas funciones que arma el endpoint real (`_estadisticas_de`, `_valor_comparado_de`, etc.), copiadas sin la capa de FastAPI. Todo lo que calculan sale de las clases y columnas reales, no de datos inventados para el demo.

In [3]:
from app.domain.feature_labels import etiqueta_legible
from app.narrativa.generador import RESPONSE_SCHEMA, SYSTEM_PROMPT, NarradorComparacion, _datos_para_prompt
from app.schemas.comparacion import (
    ArquetipoComparado,
    ComparacionJugadores,
    ContribucionComparada,
    NarrativaComparacion,
    PercentilComparado,
    ValorComparado,
)
from app.schemas.jugador import EstadisticasJugador
from app.schemas.percentiles import GrupoComparacion
from app.valuation.predictor import Predictor

PERCENTILE_STATS = [  # mismo subconjunto que api/v1/jugadores.py -- ver AGENTS.md sobre
    "Per 90 Minutes_Gls", "Per 90 Minutes_Ast", "Standard_Sh/90", "Standard_SoT%",
    "Performance_TklW", "Performance_Int", "Performance_Crs", "Playing Time_Min%",
]

predictor = Predictor(Path("../models"))


def _url_de(fila, columna):
    valor = fila.get(columna)
    return str(valor) if pd.notna(valor) else None


def _estadisticas_de(player_id, fila):
    return EstadisticasJugador(
        player_id=player_id, nombre=str(fila["Player"]), club=str(fila["Squad"]),
        nacionalidad=str(fila["Nation"]), posicion=str(fila["Pos"]),
        posicion_detallada=str(fila["posicion_tm"]), liga=str(fila["Comp"]),
        temporada=str(fila["Season"]), foto_url=_url_de(fila, "foto_url"),
        bandera_url=_url_de(fila, "bandera_url"), escudo_url=_url_de(fila, "escudo_url"),
        edad=float(fila["Age"]), minutos_jugados=int(fila["Playing Time_Min"]),
        goles=int(fila["Performance_Gls"]), asistencias=int(fila["Performance_Ast"]),
        tiros=int(fila["Standard_Sh"]), tarjetas_amarillas=int(fila["Performance_CrdY"]),
        valor_mercado_eur=float(fila["valor_eur"]),
        contrato_conocido=bool(fila["flag_contrato_conocido"]),
        dias_contrato_restante=(
            float(fila["dias_contrato_restante"]) if fila["flag_contrato_conocido"] else None
        ),
        gano_liga=bool(fila["Ganador_Liga"]), gano_champions=bool(fila["Ganador_Champions"]),
    )


def _grupo_comparacion_de(fila):
    return GrupoComparacion(
        posicion=str(fila["posicion_tm"]), temporada=str(fila["Season"]),
        n=int(fila["n_comparacion"]),
    )


def _percentiles_comparados_de(fila_a, fila_b):
    return [
        PercentilComparado(
            estadistica=stat, etiqueta=etiqueta_legible(stat),
            percentil_a=float(fila_a[f"pctl_{stat}"]), percentil_b=float(fila_b[f"pctl_{stat}"]),
            valor_bruto_a=float(fila_a[stat]), valor_bruto_b=float(fila_b[stat]),
            diferencia=float(fila_a[f"pctl_{stat}"]) - float(fila_b[f"pctl_{stat}"]),
        )
        for stat in PERCENTILE_STATS
    ]


def _valor_comparado_de(fila_a, fila_b, top_n=8):
    banda_a = predictor.predecir_banda(fila_a)
    banda_b = predictor.predecir_banda(fila_b)
    diferencias = predictor.explicar_diferencia(fila_a, fila_b, top_n=top_n)
    return ValorComparado(
        banda_a=banda_a, banda_b=banda_b,
        diferencia_valor_medio_eur=banda_a["valor_medio"] - banda_b["valor_medio"],
        contribuciones_diferencia=[ContribucionComparada(**c) for c in diferencias],
    )


def _arquetipo_comparado_de(fila_a, fila_b):
    cluster_a, cluster_b = int(fila_a["cluster_id"]), int(fila_b["cluster_id"])
    return ArquetipoComparado(
        cluster_id_a=cluster_a, etiqueta_a=str(fila_a["cluster_label"]),
        cluster_id_b=cluster_b, etiqueta_b=str(fila_b["cluster_label"]),
        mismo_arquetipo=cluster_a == cluster_b,
    )


comparacion = ComparacionJugadores(
    jugador_a=_estadisticas_de(fila_a["player_id"], fila_a),
    jugador_b=_estadisticas_de(fila_b["player_id"], fila_b),
    grupo_comparacion_a=_grupo_comparacion_de(fila_a),
    grupo_comparacion_b=_grupo_comparacion_de(fila_b),
    percentiles=_percentiles_comparados_de(fila_a, fila_b),
    valor=_valor_comparado_de(fila_a, fila_b),
    arquetipo=_arquetipo_comparado_de(fila_a, fila_b),
)

print(f"{comparacion.jugador_a.nombre}: {comparacion.valor.banda_a.valor_medio:,.0f} €")
print(f"{comparacion.jugador_b.nombre}: {comparacion.valor.banda_b.valor_medio:,.0f} €")


Erling Haaland: 332,425,329 €
Kylian Mbappé: 88,417,891 €


## 3. Qué JSON ve el LLM

`_datos_para_prompt` es el único punto de contacto entre el LLM y los datos -- nunca le pasa las estadísticas crudas, solo el resumen ya interpretado (percentiles 0-100, valores redondeados, y qué factor favorece a quién). El LLM no puede inventar una cifra que no esté acá.

In [4]:
datos = _datos_para_prompt(comparacion)
print(json.dumps(datos, indent=2, ensure_ascii=False))


{
  "jugador_a": {
    "nombre": "Erling Haaland",
    "club": "Manchester City",
    "posicion": "Centre-Forward",
    "edad": 25.0,
    "temporada": "2025-2026"
  },
  "jugador_b": {
    "nombre": "Kylian Mbappé",
    "club": "Real Madrid",
    "posicion": "Centre-Forward",
    "edad": 26.0,
    "temporada": "2025-2026"
  },
  "misma_posicion": true,
  "percentiles_vs_su_posicion": [
    {
      "estadistica": "Goles por 90'",
      "percentil_a": 95.7,
      "percentil_b": 96.5
    },
    {
      "estadistica": "Asistencias por 90'",
      "percentil_a": 88.2,
      "percentil_b": 79.4
    },
    {
      "estadistica": "Tiros por 90'",
      "percentil_a": 87.5,
      "percentil_b": 95.1
    },
    {
      "estadistica": "% de tiros a puerta",
      "percentil_a": 76.9,
      "percentil_b": 68.9
    },
    {
      "estadistica": "Entradas ganadas",
      "percentil_a": 74.1,
      "percentil_b": 26.5
    },
    {
      "estadistica": "Intercepciones",
      "percentil_a": 76.2,
    

## 4. El prompt y el schema

El `SYSTEM_PROMPT` fija el rol y, sobre todo, las restricciones: no inventar cifras, tratar el valor como estimación (nunca "recomendación de compra/venta"), y aclarar cuando los jugadores juegan posiciones distintas.

El schema de respuesta usa `json_schema` con `strict: True`, para forzar 4 campos fijos en vez de dejar al modelo redactar libre.

In [5]:
print(SYSTEM_PROMPT)


Eres un analista de scouting de fútbol. Se te entrega una comparación ya calculada entre dos jugadores: percentiles frente a jugadores de su misma posición y temporada, la contribución de un modelo de valor de mercado a la diferencia de valor entre ambos, y su arquetipo de estilo de juego (de un modelo de clustering). Tu único trabajo es REDACTAR esa comparación en español de forma clara, objetiva y CONCISA para un scout: ve directo a lo importante (fortalezas de cada uno y por qué difiere el valor), sin relleno ni frases genéricas.

Reglas estrictas:
- No inventes cifras, estadísticas ni hechos que no estén en los datos entregados.
- No calcules nada nuevo: limita tus afirmaciones a lo que los números ya indican.
- El valor de mercado es una estimación de un modelo estadístico, no una tasación oficial ni una recomendación de compra o venta: trátalo siempre como estimación.
- Si los dos jugadores juegan posiciones distintas, acláralo antes de comparar percentiles.
- Cada factor de 'fac

In [6]:
print(json.dumps(RESPONSE_SCHEMA, indent=2, ensure_ascii=False))


{
  "name": "narrativa_comparacion",
  "schema": {
    "type": "object",
    "properties": {
      "resumen": {
        "type": "string",
        "description": "1-2 oraciones que sitúan la comparación (posición y arquetipo de cada uno, y si son directamente comparables). Nada más -- el detalle va en fortalezas y en la explicación de valor, no acá."
      },
      "fortalezas_jugador_a": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "2 a 3 puntos donde el jugador A destaca sobre el B. Cada punto: una sola oración, directa, que nombre la estadística y qué implica."
      },
      "fortalezas_jugador_b": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "2 a 3 puntos donde el jugador B destaca sobre el A. Cada punto: una sola oración, directa, que nombre la estadística y qué implica."
      },
      "explicacion_diferencia_valor": {
        "type": "string",
        "desc

## 5. El bug que encontramos probando en serio

La primera versión mandaba un booleano (`favorece_a`) para decir a qué jugador beneficiaba cada factor -- el LLM tenía que traducir eso a un nombre, y en pruebas repetidas a veces se equivocaba de jugador en la prosa.

El arreglo fue sacarle esa traducción de encima: ahora mandamos directamente el nombre resuelto (`jugador_favorecido`) -- el string que tiene que copiar, no algo que tenga que derivar.

In [7]:
# Antes (causaba atribución cruzada en pruebas repetidas):
factor_antes = {"factor": "Fuerza histórica del equipo", "favorece_a": True}
# El LLM tenía que resolver: True -> "a" -> comparacion.jugador_a.nombre

# Después (lo que corre en producción, ver _datos_para_prompt arriba):
factor_despues = {"factor": "Fuerza histórica del equipo", "jugador_favorecido": "Erling Haaland"}
# Sin indirección: el LLM solo copia el nombre que ya viene resuelto.


## 6. Llamar al modelo real

Acá llamamos a `NarradorComparacion.redactar()` igual que lo haría el endpoint real.

In [8]:
narrador = NarradorComparacion(api_key=client_api_key)
resultado = narrador.redactar(comparacion)
print(json.dumps(resultado, indent=2, ensure_ascii=False))


{
  "resumen": "Erling Haaland y Kylian Mbappé, ambos delanteros centro con el arquetipo de 'estrella ofensiva de máximo volumen', son comparables directamente tanto por posición como por rol.",
  "fortalezas_jugador_a": [
    "Haaland supera a Mbappé en asistencias por 90', mostrando mayor aporte asociativo en la posición.",
    "Haaland tiene un porcentaje más alto de tiros a puerta, reflejando mejor precisión o selección de remates.",
    "Haaland destaca más en entradas ganadas e intercepciones, contribuyendo más en presión y recuperación defensiva."
  ],
  "fortalezas_jugador_b": [
    "Mbappé lidera en goles por 90', situándose en el percentil superior incluso frente a Haaland.",
    "Mbappé ejecuta más tiros por 90', indicando volumen ofensivo y protagonismo en el ataque.",
    "Mbappé tiene un altísimo percentil en centros, sumando capacidad de desborde o aporte desde banda."
  ],
  "explicacion_diferencia_valor": "La estimación de valor de mercado favorece ampliamente a Haalan

## 7. Validar contra el schema real

`NarrativaComparacion` es el mismo `response_model` del endpoint -- si lo que devuelve el LLM no calza, revienta acá, antes de llegar al frontend. También le agrega la nota de metodología, que es texto fijo del backend, nunca algo que redacte el LLM.

In [9]:
narrativa = NarrativaComparacion(**resultado)
print(narrativa.metodologia_nota)


Estimación de un modelo estadístico entrenado con datos históricos de rendimiento y valor de mercado. No es una tasación oficial ni una recomendación de compra o venta: preséntese siempre como rango, nunca como cifra puntual.


## 8. Decisiones de producto (detalle en AGENTS.md)

- Endpoint aparte, con un botón explícito en el frontend -- la llamada a OpenAI cuesta y puede tardar ~5-40s.
- Sin `OPENAI_API_KEY`, el endpoint responde 503 en vez de inventar algo.
- El LLM nunca recibe datos crudos ni calcula nada -- si algún día se agrega un campo al prompt, tiene que venir ya resuelto, nunca algo que el LLM tenga que derivar (la lección de la sección 5).